# Eval — Fine-tuned Model (`rahul042/gemma_3_lora_adapter`)

**Run this on a T4 Colab instance.**  
This notebook:
1. Installs dependencies
2. Reconstructs the exact eval split used during training (same seeds)
3. Runs the fine-tuned model on 300 eval examples
4. Reports Exact Match, Execution Accuracy, Syntax Error Rate
5. Saves results to `finetuned_results.json` for comparison in `eval_base.ipynb`

> **Note:** `rahul042/gemma_3_better` on HF is a GGUF file and cannot be loaded here.  
> We load `rahul042/gemma_3_lora_adapter` (the LoRA adapter weights) instead.

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, re, sqlite3, time, torch
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

## 1. Reconstruct Eval Split
Uses the exact same seeds as training — guaranteed identical eval set.

In [ ]:
# ── Load raw dataset ──────────────────────────────────────────────────────────
raw_dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Total examples in dataset: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")

In [ ]:
# ── Load tokenizer (needed for length filtering) ──────────────────────────────
# We need the tokenizer to replicate the <=512 token filter from training.
# Load fine-tuned model here; we'll use the same tokenizer for eval too.
model, tokenizer = FastModel.from_pretrained(
    model_name     = "rahul042/gemma_3_lora_adapter",
    max_seq_length = 512,
    load_in_4bit   = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
FastModel.for_inference(model)
print("Fine-tuned model loaded.")

In [ ]:
# ── Replicate training data prep exactly ──────────────────────────────────────
def formatting_prompts_func(examples):
    texts = []
    for question, context, sql in zip(
        examples["question"],
        examples["context"],
        examples["answer"],
    ):
        convo = [
            {
                "role": "user",
                "content": (
                    "Generate one SQL query that answers the question "
                    "using only the schema below.\n"
                    "Return only SQL. Do not include markdown, "
                    "comments, or explanation.\n\n"
                    f"Schema:\n{context}\n\n"
                    f"Question: {question}"
                ),
            },
            {"role": "model", "content": sql.strip()},
        ]
        text = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.shuffle(seed=3407).select(range(7000))   # same seed
dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=4)
dataset = dataset.filter(lambda x: len(tokenizer.encode(x["text"])) <= 512)
split   = dataset.train_test_split(test_size=0.05, seed=42)    # same seed

eval_dataset = split["test"]
print(f"Eval split size: {len(eval_dataset)} examples")
print(f"Sample question: {eval_dataset[0]['question']}")

## 2. Eval Helpers

In [ ]:
# ── SQL normalization ─────────────────────────────────────────────────────────
def normalize_sql(sql: str) -> str:
    """Lowercase, collapse whitespace, strip trailing semicolons."""
    sql = sql.strip().rstrip(";")
    # strip markdown code fences if model added them
    sql = re.sub(r"```(?:sql)?\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```", "", sql)
    sql = sql.strip()
    sql = sql.lower()
    sql = re.sub(r"\s+", " ", sql)
    # normalize INNER JOIN → JOIN
    sql = re.sub(r"\binner join\b", "join", sql)
    return sql


def extract_sql(raw_output: str) -> str:
    """Pull SQL out of model output even if it added explanation text."""
    # try to find a sql code block first
    fence = re.search(r"```(?:sql)?\s*([\s\S]+?)```", raw_output, re.IGNORECASE)
    if fence:
        return fence.group(1).strip()
    # otherwise take the first line that looks like SQL
    for line in raw_output.splitlines():
        line = line.strip()
        if re.match(r"^(SELECT|INSERT|UPDATE|DELETE|WITH|CREATE)", line, re.IGNORECASE):
            return line
    # fallback: return raw output and let execute_sql handle the error
    return raw_output.strip()


def execute_sql(schema: str, query: str):
    """Run query against in-memory SQLite. Returns frozenset of rows or None on error."""
    try:
        conn = sqlite3.connect(":memory:")
        conn.executescript(schema)
        rows = frozenset(conn.execute(query).fetchall())
        conn.close()
        return rows
    except Exception:
        return None


print("Helpers defined.")

In [ ]:
# ── Generation function ───────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Generate one SQL query that answers the question "
    "using only the schema below.\n"
    "Return only SQL. Do not include markdown, comments, or explanation."
)

def generate_sql(model, tokenizer, schema: str, question: str) -> str:
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": f"{SYSTEM_PROMPT}\n\nSchema:\n{schema}\n\nQuestion: {question}"
        }]
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0,
            do_sample=False,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_sql(raw)


print("Generation function defined.")

## 3. Run Evaluation

In [ ]:
# ── Run eval on fine-tuned model ──────────────────────────────────────────────
N_EVAL = 300  # set to len(eval_dataset) to run all

samples = eval_dataset.select(range(min(N_EVAL, len(eval_dataset))))

exact_matches = 0
exec_matches  = 0
syntax_errors = 0
wrong_results = 0
failures      = []   # store worst cases for analysis
t0            = time.time()

for i, row in enumerate(tqdm(samples, desc="fine-tuned")):
    schema   = row["context"]
    question = row["question"]
    gold     = row["answer"].strip()

    pred = generate_sql(model, tokenizer, schema, question)

    # ── exact match ──────────────────────────────────────────────────────────
    em = normalize_sql(pred) == normalize_sql(gold)
    if em:
        exact_matches += 1

    # ── execution accuracy ───────────────────────────────────────────────────
    gold_result = execute_sql(schema, gold)
    pred_result = execute_sql(schema, pred)

    if pred_result is None:
        syntax_errors += 1
        failures.append({
            "idx": i, "error": "syntax_error",
            "question": question, "schema": schema,
            "gold": gold, "pred": pred
        })
    elif pred_result == gold_result:
        exec_matches += 1
    else:
        wrong_results += 1
        failures.append({
            "idx": i, "error": "wrong_result",
            "question": question, "schema": schema,
            "gold": gold, "pred": pred
        })

total   = len(samples)
elapsed = time.time() - t0

ft_results = {
    "label":             "rahul042/gemma_3_lora_adapter (fine-tuned)",
    "n":                 total,
    "exact_match":       exact_matches / total,
    "exec_accuracy":     exec_matches  / total,
    "syntax_error_rate": syntax_errors / total,
    "wrong_result_rate": wrong_results / total,
    "elapsed_s":         round(elapsed, 1),
    "failures":          failures,
}

print(f"\n{'='*56}")
print(f"  Fine-tuned model  (n={total})")
print(f"{'='*56}")
print(f"  Exact Match:        {exact_matches:3d}/{total} = {ft_results['exact_match']:.1%}")
print(f"  Execution Accuracy: {exec_matches:3d}/{total} = {ft_results['exec_accuracy']:.1%}")
print(f"  Syntax Errors:      {syntax_errors:3d}/{total} = {ft_results['syntax_error_rate']:.1%}")
print(f"  Wrong Result:       {wrong_results:3d}/{total} = {ft_results['wrong_result_rate']:.1%}")
print(f"  Eval time:          {elapsed:.0f}s ({elapsed/total:.1f}s/sample)")

## 4. Failure Analysis

In [ ]:
# ── Categorize failure types ──────────────────────────────────────────────────
syntax_fails = [f for f in failures if f["error"] == "syntax_error"]
wrong_fails  = [f for f in failures if f["error"] == "wrong_result"]

print(f"Total failures: {len(failures)}/{total}")
print(f"  Syntax errors: {len(syntax_fails)}")
print(f"  Wrong results: {len(wrong_fails)}")

print("\n── Sample syntax errors (first 3) ──")
for f in syntax_fails[:3]:
    print(f"\nQ: {f['question']}")
    print(f"Gold: {f['gold']}")
    print(f"Pred: {f['pred']}")

print("\n── Sample wrong results (first 3) ──")
for f in wrong_fails[:3]:
    print(f"\nQ: {f['question']}")
    print(f"Gold: {f['gold']}")
    print(f"Pred: {f['pred']}")

In [ ]:
# ── Keyword-level failure breakdown ──────────────────────────────────────────
keywords = ["JOIN", "GROUP BY", "HAVING", "ORDER BY", "COUNT", "SUM", "AVG", "MAX", "MIN", "DISTINCT"]

print("── Failure rate by SQL keyword (in gold query) ──")
print(f"{'Keyword':<12} {'Fails':>6} {'Total':>6} {'Fail%':>7}")
print("-" * 35)

for kw in keywords:
    kw_samples = [r for r in samples if kw.lower() in r["answer"].lower()]
    kw_fails   = [f for f in failures if kw.lower() in f["gold"].lower()]
    if kw_samples:
        rate = len(kw_fails) / len(kw_samples)
        print(f"{kw:<12} {len(kw_fails):>6} {len(kw_samples):>6} {rate:>7.1%}")

## 5. Save Results
Save to JSON so `eval_base.ipynb` can load it and print the side-by-side comparison.

In [ ]:
# ── Save results (strip heavy failure logs for file size) ─────────────────────
save_payload = {k: v for k, v in ft_results.items() if k != "failures"}
save_payload["failure_count"] = len(failures)
save_payload["syntax_error_count"] = len(syntax_fails)
save_payload["wrong_result_count"] = len(wrong_fails)

with open("finetuned_results.json", "w") as f:
    json.dump(save_payload, f, indent=2)

print("Results saved to finetuned_results.json")
print("Download this file and upload it when running eval_base.ipynb")

# if on Colab, also copy to Drive if mounted
try:
    import shutil
    shutil.copy("finetuned_results.json", "/content/drive/MyDrive/finetuned_results.json")
    print("Also copied to Google Drive.")
except Exception:
    print("(Drive not mounted — download manually from Files panel)")

In [ ]:
# ── GPU memory summary ────────────────────────────────────────────────────────
used_gb = torch.cuda.max_memory_reserved() / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Peak VRAM used: {used_gb:.2f} GB / {total_gb:.1f} GB ({used_gb/total_gb:.1%})")